In [0]:
# ============================================================
# MVP VTI CBL
# Notebook: 04_Qualidade_Dados
#
# Objetivo:
# Consolidar as verificações de qualidade dos dados
# utilizadas no pipeline CBL/VTI.
#
# Dimensões avaliadas:
# - Completude
# - Consistência
# - Unicidade
# - Integridade espacial
# - Cobertura dos relacionamentos
# - Cobertura temporal
# ============================================================

In [0]:
# Importa as funções do PySpark utilizadas nas verificações e indicadores de qualidade dos dados
from pyspark.sql import functions as F
# Importa Window para eventuais validações analíticas realizadas ao longo do notebook
from pyspark.sql.window import Window

In [0]:
# Carrega a base CBL original da camada Bronze para comparação com os dados tratados
df_bronze_cbl = spark.table("mvp_vti.bronze.bronze_cbl_raw")
# Carrega as tabelas Silver utilizadas nas verificações de completude e consistência espacial
df_silver_cbl = spark.table("mvp_vti.silver.silver_cbl")
df_silver_locais = spark.table("mvp_vti.silver.silver_locais")
df_silver_blocos = spark.table("mvp_vti.silver.silver_blocos")
# Carrega a tabela fato Gold para validação da integração entre CBL, Locais e Blocos de 25 m
df_fato = spark.table("mvp_vti.gold.fato_cbl_bloco")
# Carrega as agregações Gold utilizadas nas verificações de cobertura e consistência temporal
df_agg_bloco = spark.table("mvp_vti.gold.agg_cbl_bloco")
df_agg_semana = spark.table("mvp_vti.gold.agg_cbl_bloco_semana")
df_degradacao = spark.table("mvp_vti.gold.agg_degradacao_bloco")

In [0]:
# Confere o volume de registros das principais tabelas utilizadas ao longo do pipeline
# A comparação permite validar a disponibilidade dos dados entre as camadas Bronze, Silver e Gold
print("=== VOLUME DAS PRINCIPAIS TABELAS ===")
print("Bronze CBL:", df_bronze_cbl.count())
print("Silver CBL:", df_silver_cbl.count())
print("Silver Locais:", df_silver_locais.count())
print("Silver Blocos:", df_silver_blocos.count())
print("Gold Fato:", df_fato.count())
print("Gold Blocos:", df_agg_bloco.count())
print("Gold Semanal:", df_agg_semana.count())
print("Gold Degradação:", df_degradacao.count())

=== VOLUME DAS PRINCIPAIS TABELAS ===
Bronze CBL: 198239
Silver CBL: 42881
Silver Locais: 797
Silver Blocos: 78160
Gold Fato: 42881
Gold Blocos: 7709
Gold Semanal: 35369
Gold Degradação: 2972


In [0]:
# Avalia a completude dos principais campos da base CBL após o tratamento na camada Silver
# Estes atributos são necessários para a localização ferroviária, análise temporal e cálculo dos indicadores CBL
display(
df_silver_cbl.select(
F.count("*").alias("total_registros"),
F.sum(F.col("linha").isNull().cast("int")).alias("nulos_linha"),
F.sum(F.col("km_major").isNull().cast("int")).alias("nulos_km"),
F.sum(F.col("kp_minor").isNull().cast("int")).alias("nulos_metro"),
F.sum(F.col("posicao_m").isNull().cast("int")).alias("nulos_posicao"),
F.sum(F.col("date_time").isNull().cast("int")).alias("nulos_data_hora"),
F.sum(F.col("cbl").isNull().cast("int")).alias("nulos_cbl")
)
)

total_registros,nulos_linha,nulos_km,nulos_metro,nulos_posicao,nulos_data_hora,nulos_cbl
42881,0,1140,1140,1140,0,0


In [0]:
# Verifica os valores mínimo e máximo de kp_minor utilizados na composição da posição ferroviária
# A análise auxilia na validação da consistência da referência KM + metro da base CBL
display(
df_silver_cbl.select(
F.min("kp_minor").alias("min_kp_minor"),
F.max("kp_minor").alias("max_kp_minor")
)
)

min_kp_minor,max_kp_minor
0,1000


In [0]:
# Identifica registros com kp_minor fora da faixa esperada para a referência KM + metro
# Valores negativos ou iguais/superiores a 1000 poderiam comprometer o cálculo da posição ferroviária
kp_invalidos = (
df_silver_cbl
.filter(
(F.col("kp_minor") < 0)
| (F.col("kp_minor") >= 1000)
)
.count()
)
# Apresenta a quantidade de registros encontrados fora da faixa esperada
print("KP Minor inválidos:", kp_invalidos)

KP Minor inválidos: 15


In [0]:
# Verifica se a posição ferroviária armazenada corresponde ao cálculo KM × 1000 + metro
# Esta validação é importante porque posicao_m é a referência utilizada nos relacionamentos espaciais da camada Gold
erros_posicao = (
df_silver_cbl
.filter(
F.col("posicao_m")
!= (F.col("km_major") * 1000 + F.col("kp_minor"))
)
.count()
)
# Apresenta a quantidade de registros com divergência no cálculo da posição ferroviária
print("Erros no cálculo de posição:", erros_posicao)

Erros no cálculo de posição: 0


In [0]:
# Verifica se existem Locais de Instalação com intervalo espacial inválido
# Para participar corretamente do relacionamento espacial, fim_m deve ser maior que inicio_m
locais_invalidos = (
df_silver_locais
.filter(F.col("fim_m") <= F.col("inicio_m"))
.count()
)
# Apresenta a quantidade de intervalos encontrados fora da regra esperada
print("Intervalos de locais inválidos:", locais_invalidos)

Intervalos de locais inválidos: 0


In [0]:
# Verifica a distribuição das extensões dos blocos após a normalização espacial realizada na Silver
display(
df_silver_blocos
.groupBy("extensao_m")
.count()
.orderBy("extensao_m")
)
# Identifica blocos cuja extensão seja diferente dos 25 m definidos para a segmentação utilizada no MVP
blocos_invalidos = (
df_silver_blocos
.filter(F.col("extensao_m") != 25)
.count()
)
# Apresenta a quantidade de blocos encontrados fora da extensão esperada
print("Blocos com extensão diferente de 25 m:", blocos_invalidos)

extensao_m,count
25,78160


Blocos com extensão diferente de 25 m: 0


In [0]:
# Verifica se existem identificadores de bloco repetidos na base Silver
# Cada bloco_id deve representar um único segmento de 25 m dentro da respectiva linha ferroviária
blocos_duplicados = (
df_silver_blocos
.groupBy("bloco_id")
.count()
.filter(F.col("count") > 1)
)
# Apresenta a quantidade de blocos com identificadores duplicados
print("Blocos duplicados:", blocos_duplicados.count())

Blocos duplicados: 0


In [0]:
# Considera o total de medições disponíveis na tabela fato Gold
total = df_fato.count()
# Conta as medições que possuem um Local de Instalação associado
com_local = (
df_fato
.filter(F.col("taxonomia").isNotNull())
.count()
)
# Conta as medições que permaneceram sem correspondência na base de Locais
sem_local = (
df_fato
.filter(F.col("taxonomia").isNull())
.count()
)
# Calcula o percentual de cobertura obtido no relacionamento espacial CBL × Locais
cobertura_local = com_local / total * 100
# Apresenta os indicadores consolidados de cobertura
print("=== COBERTURA CBL × LOCAIS ===")
print(f"Total: {total}")
print(f"Com local: {com_local}")
print(f"Sem local: {sem_local}")
print(f"Cobertura: {cobertura_local:.2f}%")

=== COBERTURA CBL × LOCAIS ===
Total: 42881
Com local: 39546
Sem local: 3335
Cobertura: 92.22%


In [0]:
# Conta as medições CBL que possuem um Bloco de 25 m associado na tabela fato
com_bloco = (
df_fato
.filter(F.col("bloco_id").isNotNull())
.count()
)
# Conta as medições que permaneceram sem correspondência na base oficial de Blocos
sem_bloco = (
df_fato
.filter(F.col("bloco_id").isNull())
.count()
)
# Calcula o percentual de cobertura do relacionamento espacial CBL × Blocos de 25 m
cobertura_bloco = com_bloco / total * 100
# Apresenta os indicadores consolidados de cobertura
print("=== COBERTURA CBL × BLOCOS 25M ===")
print(f"Total: {total}")
print(f"Com bloco: {com_bloco}")
print(f"Sem bloco: {sem_bloco}")
print(f"Cobertura: {cobertura_bloco:.2f}%")

=== COBERTURA CBL × BLOCOS 25M ===
Total: 42881
Com bloco: 41499
Sem bloco: 1382
Cobertura: 96.78%


In [0]:
# Identifica as medições CBL que possuem simultaneamente Local de Instalação e Bloco de 25 m associados
completo = (
df_fato
.filter(
F.col("taxonomia").isNotNull()
& F.col("bloco_id").isNotNull()
)
.count()
)
# Calcula o percentual de medições com enriquecimento espacial completo
cobertura_completa = completo / total * 100
# Apresenta o resultado consolidado da integração espacial das três fontes
print("=== COBERTURA ESPACIAL COMPLETA ===")
print(f"Total CBL: {total}")
print(f"Com Local + Bloco: {completo}")
print(f"Cobertura completa: {cobertura_completa:.2f}%")

=== COBERTURA ESPACIAL COMPLETA ===
Total CBL: 42881
Com Local + Bloco: 39546
Cobertura completa: 92.22%


In [0]:
# Classifica cada medição conforme a existência de associação com Local de Instalação e Bloco de 25 m
# A matriz permite visualizar de forma conjunta as diferentes situações de cobertura espacial da tabela fato
display(
df_fato
.withColumn(
"possui_local",
F.col("taxonomia").isNotNull()
)
.withColumn(
"possui_bloco",
F.col("bloco_id").isNotNull()
)
.groupBy(
"possui_local",
"possui_bloco"
)
.count()
.orderBy(
"possui_local",
"possui_bloco"
)
)

possui_local,possui_bloco,count
false,false,1382
false,true,1953
true,true,39546


In [0]:
# Verifica se as medições associadas a um bloco permanecem dentro dos limites espaciais correspondentes
# A regra deve respeitar o mesmo intervalo utilizado no relacionamento da camada Gold
erros_bloco = (
df_fato
.filter(F.col("bloco_id").isNotNull())
.filter(
(F.col("posicao_m") < F.col("bloco_inicio_m"))
| (F.col("posicao_m") >= F.col("bloco_fim_m"))
)
.count()
)
# Apresenta a quantidade de medições cuja posição não corresponde ao intervalo do bloco associado
print("Medições fora do intervalo do bloco:", erros_bloco)

Medições fora do intervalo do bloco: 0


In [0]:
# Verifica se as medições associadas a um Local permanecem dentro dos limites espaciais correspondentes
# A regra deve respeitar o mesmo intervalo utilizado no relacionamento da camada Gold
erros_local = (
df_fato
.filter(F.col("taxonomia").isNotNull())
.filter(
(F.col("posicao_m") < F.col("local_inicio_m"))
| (F.col("posicao_m") >= F.col("local_fim_m"))
)
.count()
)
# Apresenta a quantidade de medições cuja posição não corresponde ao intervalo do Local associado
print("Medições fora do intervalo do local:", erros_local)

Medições fora do intervalo do local: 0


In [0]:
# Verifica se existem IDs repetidos na tabela fato após os relacionamentos espaciais da camada Gold
# Caso sejam encontradas duplicidades, o resultado deve ser investigado antes de qualquer exclusão de registros
ids_duplicados = (
df_fato
.groupBy("id")
.count()
.filter(F.col("count") > 1)
)
# Apresenta a quantidade de IDs que aparecem mais de uma vez na tabela fato
print("IDs duplicados na fato:", ids_duplicados.count())

IDs duplicados na fato: 0


In [0]:
# Avalia a cobertura temporal disponível na tabela fato a partir das datas das medições CBL
# A validação registra a quantidade de dias com dados e os limites inicial e final do período analisado
display(
df_fato.select(
F.countDistinct(F.to_date("date_time")).alias("qtd_dias"),
F.min(F.to_date("date_time")).alias("primeira_data"),
F.max(F.to_date("date_time")).alias("ultima_data")
)
)

qtd_dias,primeira_data,ultima_data
277,2025-03-15,2026-06-11


In [0]:
# Consolida o histórico temporal disponível para cada Bloco de 25 m
# A quantidade de semanas observadas permite avaliar a disponibilidade de dados para análise de tendência
df_cobertura_temporal = (
df_agg_semana
.groupBy(
"linha",
"bloco_id"
)
.agg(
F.countDistinct("semana").alias("qtd_semanas"),
F.min("semana").alias("primeira_semana"),
F.max("semana").alias("ultima_semana")
)
)
# Exibe os blocos com maior quantidade de semanas disponíveis no histórico
display(
df_cobertura_temporal
.orderBy(F.desc("qtd_semanas"))
.limit(50)
)

linha,bloco_id,qtd_semanas,primeira_semana,ultima_semana
LP002,LP002_858175_858200,37,2025-03-10T00:00:00.000Z,2026-05-25T00:00:00.000Z
LP002,LP002_53600_53625,37,2025-03-10T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_830325_830350,36,2025-03-10T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_830300_830325,34,2025-03-10T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_858325_858350,33,2025-03-10T00:00:00.000Z,2026-05-25T00:00:00.000Z
LP002,LP002_551675_551700,33,2025-04-14T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_346950_346975,32,2025-03-10T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_764700_764725,32,2025-03-17T00:00:00.000Z,2026-06-01T00:00:00.000Z
LP002,LP002_347175_347200,31,2025-04-07T00:00:00.000Z,2026-06-08T00:00:00.000Z
LP002,LP002_310075_310100,31,2025-03-24T00:00:00.000Z,2026-05-04T00:00:00.000Z


In [0]:
# Conta o total de blocos que possuem algum histórico disponível na agregação semanal
total_blocos = df_cobertura_temporal.count()
# Identifica os blocos que atendem ao critério mínimo de quatro semanas para análise de tendência
blocos_suficientes = (
df_cobertura_temporal
.filter(F.col("qtd_semanas") >= 4)
.count()
)
# Identifica os blocos que não possuem histórico mínimo suficiente para o cálculo da tendência
blocos_insuficientes = (
df_cobertura_temporal
.filter(F.col("qtd_semanas") < 4)
.count()
)
# Calcula a cobertura dos blocos aptos à análise temporal
cobertura_tendencia = blocos_suficientes / total_blocos * 100
print("=== COBERTURA TEMPORAL PARA TENDÊNCIA ===")
print(f"Total de blocos: {total_blocos}")
print(f"Com >= 4 semanas: {blocos_suficientes}")
print(f"Com < 4 semanas: {blocos_insuficientes}")
print(f"Cobertura: {cobertura_tendencia:.2f}%")
# Confere quantos blocos tiveram os indicadores de tendência efetivamente calculados na camada Gold
print("Blocos com tendência calculada:", df_degradacao.count())
# Verifica a quantidade mínima, máxima e média de semanas utilizadas nos blocos com tendência calculada
display(
df_degradacao.select(
F.min("qtd_semanas").alias("min_semanas"),
F.max("qtd_semanas").alias("max_semanas"),
F.avg("qtd_semanas").alias("media_semanas")
)
)

=== COBERTURA TEMPORAL PARA TENDÊNCIA ===
Total de blocos: 7709
Com >= 4 semanas: 2972
Com < 4 semanas: 4737
Cobertura: 38.55%
Blocos com tendência calculada: 2972


min_semanas,max_semanas,media_semanas
4,37,9.422611036339166
